In [ ]:
! pip install pypsa highspy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import sys

PROJECT_DIR = "drive/MyDrive/Colab_Notebooks/PyPSA/Zonal_model/"
file_name = PROJECT_DIR + 'Nodal_vs_Zonal.pdf'

# Nodal vs Zonal Models and Croatia Zones

In [ ]:
# Updated Dictionary for the three specific zones
final_zone_data = {
    "Serbia_Pancevo": {
        "SMR_capacity": 150,           # MW (Refinery scale)
        "SMR_marginal_cost": 55,       # Based on domestic gas prices
        "Storage_capacity": 40,        # MWh (Small tank buffer)
        "H2_demand": [120, 120, 120, 120], # Constant refinery load
        "Location": "Inland / Refinery"
    },
    "Croatia_Kutina": {
        "SMR_capacity": 120,           # MW (Large scale for Ammonia)
        "SMR_marginal_cost": 15,       # Includes estimated Carbon Tax/ETS
        "Storage_capacity": 150,       # Larger buffer for ammonia synthesis
        "H2_demand": [140, 140, 110, 110],
        "Location": "Inland / Fertilizer"
    },
    "Croatia_Rijeka": {
        "SMR_capacity": 300,            # MW (Refinery desulfurization)
        "SMR_marginal_cost": 5,       # Includes estimated Carbon Tax/ETS
        "Storage_capacity": 500,       # Salt cavern or large-scale tankage
        "H2_demand": [70, 70, 50, 50],
        "Location": "Coastal / Port"
    },
    "Croatia": {
        "pipeline_capacity": 300,
        "CH4_price" : [30, 50, 60, 40]
    }
}

## Case 1: Large SMR in Rijeka & High Pipeline Capacity


In [ ]:
import pypsa
import pandas as pd

# Network Model
n = pypsa.Network()
n.set_snapshots(range(4))

# Electricity
n.add("Bus", "Electricity_Grid", carrier="AC")
n.add("Generator", "Grid_Supply", bus="Electricity_Grid", p_nom=1000, marginal_cost=50, carrier="AC")

# Gas Grid
n.add("Bus", "Gas_Grid", carrier="gas")
ch4_price_series = pd.Series(final_zone_data["Croatia"]["CH4_price"], index=n.snapshots)
n.add("Generator", "Gas_Source_Croatia", bus="Gas_Grid", p_nom=10000, marginal_cost=ch4_price_series, carrier="gas")

# Serbia_Pancevo
data = final_zone_data["Serbia_Pancevo"]
n.add("Bus", "Serbia_Pancevo_H2", carrier="H2")
n.add("Generator", "SMR_Pancevo", bus="Serbia_Pancevo_H2", p_nom=data["SMR_capacity"], marginal_cost=data["SMR_marginal_cost"], carrier="H2", control='PQ')
n.add("Load", "Demand_Pancevo", bus="Serbia_Pancevo_H2", p_set=pd.Series(data["H2_demand"], index=n.snapshots), carrier="H2")
n.add("Generator", "Emergency_H2_Supply_Pancevo", bus="Serbia_Pancevo_H2", p_nom=1000, marginal_cost=1000, carrier="H2", control='Slack')

# Croatia_- node Kutina
data = final_zone_data["Croatia_Kutina"]
n.add("Bus", "Croatia_Kutina_H2", carrier="H2")
n.add("Link", "SMR_Kutina", bus0="Gas_Grid", bus1="Croatia_Kutina_H2", p_nom=data["SMR_capacity"], marginal_cost=data["SMR_marginal_cost"], carrier="H2")
n.add("Load", "Demand_Kutina", bus="Croatia_Kutina_H2", p_set=pd.Series(data["H2_demand"], index=n.snapshots), carrier="H2")
n.add("Generator", "Emergency_H2_Supply_Kutina", bus="Croatia_Kutina_H2", p_nom=1000, marginal_cost=1000, carrier="H2", control='Slack')

# Croatia_- node Rijeka
data = final_zone_data["Croatia_Rijeka"]
n.add("Bus", "Croatia_Rijeka_H2", carrier="H2")
n.add("Link", "SMR_Rijeka", bus0="Gas_Grid", bus1="Croatia_Rijeka_H2", p_nom=data["SMR_capacity"], marginal_cost=data["SMR_marginal_cost"], carrier="H2")
n.add("Load", "Demand_Rijeka", bus="Croatia_Rijeka_H2", p_set=pd.Series(data["H2_demand"], index=n.snapshots), carrier="H2")
n.add("Generator", "Emergency_H2_Supply_Rijeka", bus="Croatia_Rijeka_H2", p_nom=1000, marginal_cost=1000, carrier="H2", control='Slack')

# Pipeline
n.add("Link", "Pipeline_HR", bus0="Croatia_Rijeka_H2", bus1="Croatia_Kutina_H2", p_nom=final_zone_data["Croatia"]["pipeline_capacity"], efficiency=1.0, carrier="H2")

# Sanitize the network to ensure all carriers are registered
n.sanitize()

# SOLVE
n.optimize(solver_name='highs', include_objective_constant=False)

('ok', 'optimal')

In [ ]:
# RESULTS
print("Optimization Results:")
print("----------------------")
print(f"Total system cost: {n.objective:.2f} EUR/hour")
print(f"Pipeline capacity: {n.links.p_nom.get('Pipeline_HR', 0.0):.2f} MW")

# Get average pipeline dispatch correctly using p0
pipeline_dispatch_hr_mean = 0.0
# Access p0 (flow from bus0) for Pipeline_HR and calculate its mean
pipeline_dispatch_hr_mean = n.links_t.p0['Pipeline_HR'].mean()

print(f"Average H2 flow through Pipeline_HR: {pipeline_dispatch_hr_mean:.2f} MW")

Optimization Results:
----------------------
Total system cost: 62900.00 EUR/hour
Pipeline capacity: 300.00 MW
Average H2 flow through Pipeline_HR: 125.00 MW


In [ ]:
import pandas as pd

print("\n--- H2 Balance Per Zone ---")
print("--------------------------------------------------\n")

# Get all dispatch data, filling missing columns with 0 for easier summation
gen_p = n.generators_t.p.fillna(0)
load_p = n.loads_t.p.fillna(0)
link_p0 = n.links_t.p0.fillna(0)
link_p1 = n.links_t.p1.fillna(0)

# --- Serbia_Pancevo Zone Balance ---
pancevo_h2_bus_name = "Serbia_Pancevo_H2"

pancevo_production_smr = gen_p.get('SMR_Pancevo', pd.Series(0, index=n.snapshots))
pancevo_production_emergency = gen_p.get('Emergency_H2_Supply_Pancevo', pd.Series(0, index=n.snapshots))
pancevo_total_production = pancevo_production_smr + pancevo_production_emergency

pancevo_demand = load_p.get('Demand_Pancevo', pd.Series(0, index=n.snapshots))

pancevo_balance_df = pd.DataFrame({
    'SMR_Production': pancevo_production_smr,
    'Emergency_Supply': pancevo_production_emergency,
    'Demand': pancevo_demand,
})
print(f"Zone: Serbia_Pancevo (Bus: {pancevo_h2_bus_name})\n{pancevo_balance_df.round(2)}\n")


# --- Croatia_Rijeka Zone Balance ---
rijeka_h2_bus_name = "Croatia_Rijeka_H2"

# SMR_Rijeka is a Link, p1 represents net injection at bus1 (H2 bus), so production is -p1
rijeka_production_smr = -link_p1.get('SMR_Rijeka', pd.Series(0, index=n.snapshots))
rijeka_production_emergency = gen_p.get('Emergency_H2_Supply_Rijeka', pd.Series(0, index=n.snapshots))

# Pipeline_HR is from Rijeka_H2 (bus0) to Kutina_H2 (bus1)
# Outgoing flow from Rijeka is p0 for Pipeline_HR (positive values)
rijeka_outgoing_flow = link_p0.get('Pipeline_HR', pd.Series(0, index=n.snapshots))

rijeka_balance_df = pd.DataFrame({
    'SMR_Production': rijeka_production_smr,
    'Emergency_Supply': rijeka_production_emergency,
    'Pipeline_Flow': -rijeka_outgoing_flow, # Display as negative for outflow
    'Demand': load_p.get('Demand_Rijeka', pd.Series(0, index=n.snapshots)),
})
print(f"Zone: Croatia_Rijeka (Bus: {rijeka_h2_bus_name})\n{rijeka_balance_df.round(2)}\n")


# --- Croatia_Kutina Zone Balance ---
kutina_h2_bus_name = "Croatia_Kutina_H2"

# SMR_Kutina is a Link, p1 represents net injection at bus1 (H2 bus), so production is -p1
kutina_production_smr = -link_p1.get('SMR_Kutina', pd.Series(0, index=n.snapshots))
kutina_production_emergency = gen_p.get('Emergency_H2_Supply_Kutina', pd.Series(0, index=n.snapshots))

# Pipeline_HR is from Rijeka_H2 (bus0) to Kutina_H2 (bus1)
# Incoming flow to Kutina is p1 for Pipeline_HR (negative values for inflow)
kutina_incoming_flow = link_p1.get('Pipeline_HR', pd.Series(0, index=n.snapshots))

kutina_balance_df = pd.DataFrame({
    'SMR_Production': kutina_production_smr,
    'Emergency_Supply': kutina_production_emergency,
    'Pipeline_Flow': -kutina_incoming_flow, # Display as positive for inflow
    'Demand': load_p.get('Demand_Kutina', pd.Series(0, index=n.snapshots)),
})
print(f"Zone: Croatia_Kutina (Bus: {kutina_h2_bus_name})\n{kutina_balance_df.round(2)}\n")


--- H2 Balance Per Zone ---
--------------------------------------------------

Zone: Serbia_Pancevo (Bus: Serbia_Pancevo_H2)
          SMR_Production  Emergency_Supply  Demand
snapshot                                          
0                  120.0              -0.0   120.0
1                  120.0              -0.0   120.0
2                  120.0              -0.0   120.0
3                  120.0              -0.0   120.0

Zone: Croatia_Rijeka (Bus: Croatia_Rijeka_H2)
          SMR_Production  Emergency_Supply  Pipeline_Flow  Demand
snapshot                                                         
0                  210.0              -0.0         -140.0    70.0
1                  210.0              -0.0         -140.0    70.0
2                  160.0              -0.0         -110.0    50.0
3                  160.0              -0.0         -110.0    50.0

Zone: Croatia_Kutina (Bus: Croatia_Kutina_H2)
          SMR_Production  Emergency_Supply  Pipeline_Flow  Demand
snapshot   

In [ ]:
print("\n--- Marginal Prices at H2 Buses Per Snapshot (EUR/MWh) ---")
print("----------------------------------------------------------\n")

# Filter for H2 buses only and select marginal prices
h2_bus_marginal_prices = n.buses_t.marginal_price.filter(regex='_H2')

# Rename columns for better readability (remove '_H2')
h2_bus_marginal_prices.columns = h2_bus_marginal_prices.columns.str.replace('_H2', '')

display(h2_bus_marginal_prices.round(2))


--- Marginal Prices at H2 Buses Per Snapshot (EUR/MWh) ---
----------------------------------------------------------



name,Serbia_Pancevo,Croatia_Kutina,Croatia_Rijeka
snapshot,,,
0,55.0,35.0,35.0
1,55.0,55.0,55.0
2,55.0,65.0,65.0
3,55.0,45.0,45.0


**Case Summary: Large SMR in Rijeka & High Pipeline Capacity (Case 1)**

**1. Key Parameters:**
* **Croatia_Rijeka SMR Capacity:** 300 MW (Cost: 5 EUR/MWh + Gas).
* **Croatia_Kutina SMR Capacity:** 120 MW (Cost: 15 EUR/MWh + Gas).
* **Pipeline Capacity:** 300 MW (Unconstrained).
* **Kutina Demand Profile:** [140, 140, 110, 110] MW.

**2. Optimization Results:**
* **Total System Cost:** 62,900.00 EUR/hour.
* **Pipeline Utilization:** The **Average H2 flow through Pipeline_HR is 125.00 MW**. The pipeline comfortably handles the export from Rijeka to Kutina.

**3. H2 Balance:**
* **Croatia_Rijeka:** Produces 210 MW (Snapshots 0,1) and 160 MW (Snapshots 2,3). It meets local demand and exports the remainder.
* **Croatia_Kutina:** Local SMR production is **0.0 MW**. It imports its entire requirement via the pipeline because Rijeka is cheaper.

**4. Marginal Prices:**
* Prices are identical in Rijeka and Kutina (35, 55, 65, 45 EUR/MWh), indicating a perfectly integrated market.

**In Conclusion**:

This case demonstrates how a large SMR capacity in one zone (Rijeka) combined with a high-capacity, low-cost pipeline can significantly impact the supply strategy for an interconnected zone (Kutina). Kutina effectively 'shuts down' its own SMR (due to its higher local production cost) and procures all its hydrogen via the pipeline from Rijeka, leveraging Rijeka's more cost-efficient production. The result is a **unified marginal price for hydrogen across the connected Croatian zones**, indicating efficient resource allocation facilitated by robust transmission infrastructure.

## Case 2: Decreasing H2 Transmission Capacity
Transmission capacity is set up on 80 MW

In [ ]:
n.model.solver_model = None
n2 = n.copy()
n2.links.loc["Pipeline_HR", "p_nom"] = 80
n2.optimize(solver_name='highs', include_objective_constant=False)
print(f"Case 2 Total System Cost: {n2.objective:.2f} EUR/hour")

Case 2 Total System Cost: 64700.00 EUR/hour


In [ ]:
import pandas as pd

print("\n--- H2 Balance Per Zone (All Four Snapshots) - Case 2 ---")
print("----------------------------------------------------------\n")

# Get all dispatch data from n2, filling missing columns with 0
gen_p = n2.generators_t.p.fillna(0)
load_p = n2.loads_t.p.fillna(0)
link_p0 = n2.links_t.p0.fillna(0)
link_p1 = n2.links_t.p1.fillna(0)

# --- Serbia_Pancevo Zone Balance ---
pancevo_h2_bus_name = "Serbia_Pancevo_H2"
pancevo_production_smr = gen_p.get('SMR_Pancevo', pd.Series(0, index=n2.snapshots))
pancevo_production_emergency = gen_p.get('Emergency_H2_Supply_Pancevo', pd.Series(0, index=n2.snapshots))
pancevo_demand = load_p.get('Demand_Pancevo', pd.Series(0, index=n2.snapshots))

pancevo_balance_df = pd.DataFrame({
    'SMR_Production': pancevo_production_smr,
    'Emergency_Supply': pancevo_production_emergency,
    'Demand': pancevo_demand,
})
print(f"Zone: Serbia_Pancevo (Bus: {pancevo_h2_bus_name})\n{pancevo_balance_df.round(2)}\n")

# --- Croatia_Rijeka Zone Balance ---
rijeka_h2_bus_name = "Croatia_Rijeka_H2"
rijeka_production_smr = -link_p1.get('SMR_Rijeka', pd.Series(0, index=n2.snapshots))
rijeka_production_emergency = gen_p.get('Emergency_H2_Supply_Rijeka', pd.Series(0, index=n2.snapshots))
rijeka_outgoing_flow = link_p0.get('Pipeline_HR', pd.Series(0, index=n2.snapshots))

rijeka_balance_df = pd.DataFrame({
    'SMR_Production': rijeka_production_smr,
    'Emergency_Supply': rijeka_production_emergency,
    'Pipeline_Flow': -rijeka_outgoing_flow, # Outflow is negative
    'Demand': load_p.get('Demand_Rijeka', pd.Series(0, index=n2.snapshots)),
})
print(f"Zone: Croatia_Rijeka (Bus: {rijeka_h2_bus_name})\n{rijeka_balance_df.round(2)}\n")

# --- Croatia_Kutina Zone Balance ---
kutina_h2_bus_name = "Croatia_Kutina_H2"
kutina_production_smr = -link_p1.get('SMR_Kutina', pd.Series(0, index=n2.snapshots))
kutina_production_emergency = gen_p.get('Emergency_H2_Supply_Kutina', pd.Series(0, index=n2.snapshots))
kutina_incoming_flow = link_p1.get('Pipeline_HR', pd.Series(0, index=n2.snapshots))

kutina_balance_df = pd.DataFrame({
    'SMR_Production': kutina_production_smr,
    'Emergency_Supply': kutina_production_emergency,
    'Pipeline_Flow': -kutina_incoming_flow, # Inflow is positive (-p1)
    'Demand': load_p.get('Demand_Kutina', pd.Series(0, index=n2.snapshots)),
})
print(f"Zone: Croatia_Kutina (Bus: {kutina_h2_bus_name})\n{kutina_balance_df.round(2)}\n")


--- H2 Balance Per Zone (All Four Snapshots) - Case 2 ---
----------------------------------------------------------

Zone: Serbia_Pancevo (Bus: Serbia_Pancevo_H2)
          SMR_Production  Emergency_Supply  Demand
snapshot                                          
0                  120.0              -0.0   120.0
1                  120.0              -0.0   120.0
2                  120.0              -0.0   120.0
3                  120.0              -0.0   120.0

Zone: Croatia_Rijeka (Bus: Croatia_Rijeka_H2)
          SMR_Production  Emergency_Supply  Pipeline_Flow  Demand
snapshot                                                         
0                  150.0              -0.0          -80.0    70.0
1                  150.0              -0.0          -80.0    70.0
2                  130.0              -0.0          -80.0    50.0
3                  130.0              -0.0          -80.0    50.0

Zone: Croatia_Kutina (Bus: Croatia_Kutina_H2)
          SMR_Production  Emergency_Sup

In [ ]:
print("\n--- Marginal Prices at H2 Buses Per Snapshot (EUR/MWh) - Case 2 ---")
print("------------------------------------------------------------------\n")

# Accessing n2 marginal prices specifically for H2 buses
h2_prices_n2 = n2.buses_t.marginal_price.filter(regex='_H2')
h2_prices_n2.columns = h2_prices_n2.columns.str.replace('_H2', '')

display(h2_prices_n2.round(2))


--- Marginal Prices at H2 Buses Per Snapshot (EUR/MWh) - Case 2 ---
------------------------------------------------------------------



name,Serbia_Pancevo,Croatia_Kutina,Croatia_Rijeka
snapshot,,,
0,55.0,45.0,35.0
1,55.0,65.0,55.0
2,55.0,75.0,65.0
3,55.0,55.0,45.0


**Case Summary: Decreased H2 Transmission Capacity (Case 2)**

**1. Key Parameters:**
* **Pipeline Capacity:** Reduced to **80 MW** (Constrained).
* **Kutina Demand Profile:** [140, 140, 110, 110] MW.

**2. Optimization Results:**
* **Total System Cost:** 64,700.00 EUR/hour (An increase of 1,800 EUR compared to Case 1 due to congestion).
* **Pipeline Utilization:** The pipeline is **congested at 80 MW** across all snapshots.

**3. H2 Balance:**
* **Croatia_Kutina:** Because imports are capped at 80 MW, Kutina's local SMR must activate to cover the deficit (**60 MW** in snapshots 0,1 and **30 MW** in snapshots 2,3).

**4. Marginal Prices:**
* A price separation exists between Rijeka and Kutina. While Rijeka stays at the lower cost tier (35-65 EUR/MWh), Kutina's price is set by its own more expensive local SMR (45-75 EUR/MWh), leading to a 10 EUR/MWh premium.

In [ ]:
import pandas as pd

# Calculate Price Difference (Kutina - Rijeka)
# In Case 2, Kutina is more expensive due to the 80MW constraint
price_diff = n2.buses_t.marginal_price['Croatia_Kutina_H2'] - n2.buses_t.marginal_price['Croatia_Rijeka_H2']

# Congestion Rent = Flow * Price Difference
# We use p0 for outgoing flow from Rijeka
flow = n2.links_t.p0['Pipeline_HR']
congestion_rent = flow * price_diff

rent_summary = pd.DataFrame({
    'Rijeka_Price': n2.buses_t.marginal_price['Croatia_Rijeka_H2'],
    'Kutina_Price': n2.buses_t.marginal_price['Croatia_Kutina_H2'],
    'Price_Spread': price_diff,
    'Pipeline_Flow': flow,
    'Congestion_Rent': congestion_rent
}, index=n2.snapshots)

print("--- Congestion Rent Analysis (Case 2: 80MW Constrained) ---")
display(rent_summary.round(2))
print(f"\nTotal Congestion Rent for the period: {congestion_rent.sum():.2f} EUR")

--- Congestion Rent Analysis (Case 2: 80MW Constrained) ---


,Rijeka_Price,Kutina_Price,Price_Spread,Pipeline_Flow,Congestion_Rent
snapshot,,,,,
0,35.0,45.0,10.0,80.0,800.0
1,55.0,65.0,10.0,80.0,800.0
2,65.0,75.0,10.0,80.0,800.0
3,45.0,55.0,10.0,80.0,800.0



Total Congestion Rent for the period: 3200.00 EUR


In [ ]:
import pandas as pd

# --- Case 2: Detailed Quantities, Prices, and Cash Flows ---

# 1. Rijeka Data
rijeka_prices = n2.buses_t.marginal_price['Croatia_Rijeka_H2']
rijeka_prod = -n2.links_t.p1['SMR_Rijeka']
rijeka_dem = n2.loads_t.p['Demand_Rijeka']
rijeka_rev = rijeka_prod * rijeka_prices
rijeka_pay = rijeka_dem * rijeka_prices

# 2. Kutina Data
kutina_prices = n2.buses_t.marginal_price['Croatia_Kutina_H2']
kutina_prod = -n2.links_t.p1['SMR_Kutina']
kutina_dem = n2.loads_t.p['Demand_Kutina']
kutina_rev = kutina_prod * kutina_prices
kutina_pay = kutina_dem * kutina_prices

# Combine into a comprehensive DataFrame
case2_extended = pd.DataFrame({
    'Rijeka_Price [EUR/MWh]': rijeka_prices,
    'Rijeka_Prod [MW]': rijeka_prod,
    'Rijeka_Gen_Rev [EUR]': rijeka_rev,
    'Rijeka_Con_Pay [EUR]': rijeka_pay,
    'Kutina_Price [EUR/MWh]': kutina_prices,
    'Kutina_Prod [MW]': kutina_prod,
    'Kutina_Gen_Rev [EUR]': kutina_rev,
    'Kutina_Con_Pay [EUR]': kutina_pay
}, index=n2.snapshots)

print("--- Case 2: Detailed Snapshot Financials & Quantities ---")
display(case2_extended.round(2))

# Verification of Congestion Rent via Totals
total_gen_rev = rijeka_rev.sum() + kutina_rev.sum()
total_con_pay = rijeka_pay.sum() + kutina_pay.sum()
print(f"\nTotal Consumer Payment: {total_con_pay:.2f} EUR")
print(f"Total Generator Revenue: {total_gen_rev:.2f} EUR")
print(f"Difference (Congestion Rent): {total_con_pay - total_gen_rev:.2f} EUR")

--- Case 2: Detailed Snapshot Financials & Quantities ---


,Rijeka_Price [EUR/MWh],Rijeka_Prod [MW],Rijeka_Gen_Rev [EUR],Rijeka_Con_Pay [EUR],Kutina_Price [EUR/MWh],Kutina_Prod [MW],Kutina_Gen_Rev [EUR],Kutina_Con_Pay [EUR]
snapshot,,,,,,,,
0,35.0,150.0,5250.0,2450.0,45.0,60.0,2700.0,6300.0
1,55.0,150.0,8250.0,3850.0,65.0,60.0,3900.0,9100.0
2,65.0,130.0,8450.0,3250.0,75.0,30.0,2250.0,8250.0
3,45.0,130.0,5850.0,2250.0,55.0,30.0,1650.0,6050.0



Total Consumer Payment: 41500.00 EUR
Total Generator Revenue: 38300.00 EUR
Difference (Congestion Rent): 3200.00 EUR


## Case 3: No Tranmission Line in Croatia

In [ ]:
n.model.solver_model = None
n3 = n.copy()
n3.links.loc["Pipeline_HR", "p_nom"] = 0
n3.optimize(solver_name='highs', include_objective_constant=False)
print(f"Case 3 Total System Cost: {n3.objective:.2f} EUR/hour")

Case 3 Total System Cost: 105700.00 EUR/hour


In [ ]:
import pandas as pd

print("\n--- H2 Balance Per Zone (All Four Snapshots) - Case 3 (Islanded) ---")
print("----------------------------------------------------------------------\n")

# Get all dispatch data from n3
gen_p = n3.generators_t.p.fillna(0)
load_p = n3.loads_t.p.fillna(0)
link_p0 = n3.links_t.p0.fillna(0)
link_p1 = n3.links_t.p1.fillna(0)

# --- Serbia_Pancevo Zone Balance ---
pancevo_h2_bus_name = "Serbia_Pancevo_H2"
pancevo_production_smr = gen_p.get('SMR_Pancevo', pd.Series(0, index=n3.snapshots))
pancevo_production_emergency = gen_p.get('Emergency_H2_Supply_Pancevo', pd.Series(0, index=n3.snapshots))

pancevo_balance_df = pd.DataFrame({
    'SMR_Production': pancevo_production_smr,
    'Emergency_Supply': pancevo_production_emergency,
    'Demand': load_p.get('Demand_Pancevo', pd.Series(0, index=n3.snapshots)),
})
print(f"Zone: Serbia_Pancevo (Bus: {pancevo_h2_bus_name})\n{pancevo_balance_df.round(2)}\n")

# --- Croatia_Rijeka Zone Balance ---
rijeka_h2_bus_name = "Croatia_Rijeka_H2"
rijeka_production_smr = -link_p1.get('SMR_Rijeka', pd.Series(0, index=n3.snapshots))
rijeka_production_emergency = gen_p.get('Emergency_H2_Supply_Rijeka', pd.Series(0, index=n3.snapshots))

rijeka_balance_df = pd.DataFrame({
    'SMR_Production': rijeka_production_smr,
    'Emergency_Supply': rijeka_production_emergency,
    'Pipeline_Flow': 0.0, # Isolated
    'Demand': load_p.get('Demand_Rijeka', pd.Series(0, index=n3.snapshots)),
})
print(f"Zone: Croatia_Rijeka (Bus: {rijeka_h2_bus_name})\n{rijeka_balance_df.round(2)}\n")

# --- Croatia_Kutina Zone Balance ---
kutina_h2_bus_name = "Croatia_Kutina_H2"
kutina_production_smr = -link_p1.get('SMR_Kutina', pd.Series(0, index=n3.snapshots))
kutina_production_emergency = gen_p.get('Emergency_H2_Supply_Kutina', pd.Series(0, index=n3.snapshots))

kutina_balance_df = pd.DataFrame({
    'SMR_Production': kutina_production_smr,
    'Emergency_Supply': kutina_production_emergency,
    'Pipeline_Flow': 0.0, # Isolated
    'Demand': load_p.get('Demand_Kutina', pd.Series(0, index=n3.snapshots)),
})
print(f"Zone: Croatia_Kutina (Bus: {kutina_h2_bus_name})\n{kutina_balance_df.round(2)}\n")


--- H2 Balance Per Zone (All Four Snapshots) - Case 3 (Islanded) ---
----------------------------------------------------------------------

Zone: Serbia_Pancevo (Bus: Serbia_Pancevo_H2)
          SMR_Production  Emergency_Supply  Demand
snapshot                                          
0                  120.0              -0.0   120.0
1                  120.0              -0.0   120.0
2                  120.0              -0.0   120.0
3                  120.0              -0.0   120.0

Zone: Croatia_Rijeka (Bus: Croatia_Rijeka_H2)
          SMR_Production  Emergency_Supply  Pipeline_Flow  Demand
snapshot                                                         
0                   70.0              -0.0            0.0    70.0
1                   70.0              -0.0            0.0    70.0
2                   50.0              -0.0            0.0    50.0
3                   50.0              -0.0            0.0    50.0

Zone: Croatia_Kutina (Bus: Croatia_Kutina_H2)
          SMR_Pr

In [ ]:
print("\n--- Marginal Prices at H2 Buses Per Snapshot (EUR/MWh) - Case 3 (Islanded) ---")
print("------------------------------------------------------------------\n")

# Accessing n3 marginal prices specifically for H2 buses
h2_prices_n3 = n3.buses_t.marginal_price.filter(regex='_H2')
h2_prices_n3.columns = h2_prices_n3.columns.str.replace('_H2', '')

display(h2_prices_n3.round(2))


--- Marginal Prices at H2 Buses Per Snapshot (EUR/MWh) - Case 3 (Islanded) ---
------------------------------------------------------------------



name,Serbia_Pancevo,Croatia_Kutina,Croatia_Rijeka
snapshot,,,
0,55.0,1000.0,35.0
1,55.0,1000.0,55.0
2,55.0,75.0,65.0
3,55.0,55.0,45.0


**Summary: No Transmission Line in Croatia (Case 3)**

**1. Key Parameters:**
* **Pipeline Capacity:** set to **0 MW** (Complete Isolation).
* **Kutina SMR Capacity:** 120 MW.
* **Kutina Peak Demand:** 140 MW (Snapshots 0 & 1).
* **Emergency Supply Cost:** 1,000 EUR/MWh.

**2. Optimization Results:**
* **Total System Cost:** 105,700.00 EUR/hour.
* **Marginal Prices:** Kutina's price hits **1,000 EUR/MWh** during peak times (Snapshots 0, 1) due to local shortages.

**3. H2 Balance:**
* **Croatia_Rijeka:** Produces only its local demand (70/50 MW). Its surplus capacity is stranded.
* **Croatia_Kutina:** Local SMR is maxed at **120 MW**. It relies on **20 MW of Emergency H2 Supply** during peak periods to meet the 140 MW load.

**Conclusion:** Case 3 demonstrates the high economic risk of 'energy islands.' Without interconnection, Kutina faces extreme price spikes and supply deficits even while Rijeka has idle, low-cost capacity nearby.

## Case 4: Zonal Model in Croatia
In this case, Croatia is modeled as a single zone. We aggregate the capacities and demands of Rijeka and Kutina into one bus to simulate a perfectly integrated domestic market with no internal transmission constraints.

In [ ]:
n4 = pypsa.Network()
n4.set_snapshots(range(4))

# Electricity
n4.add("Bus", "Electricity_Grid", carrier="AC")
n4.add("Generator", "Grid_Supply", bus="Electricity_Grid", p_nom=1000, marginal_cost=50, carrier="AC")

# Gas Grid
n4.add("Bus", "Gas_Grid", carrier="gas")
ch4_price_series = pd.Series(final_zone_data["Croatia"]["CH4_price"], index=n4.snapshots)
n4.add("Generator", "Gas_Source_Croatia", bus="Gas_Grid", p_nom=10000, marginal_cost=ch4_price_series, carrier="gas")

# Serbia_Pancevo (Isolated)
data_p = final_zone_data["Serbia_Pancevo"]
n4.add("Bus", "Serbia_Pancevo_H2", carrier="H2")
n4.add("Generator", "SMR_Pancevo", bus="Serbia_Pancevo_H2", p_nom=data_p["SMR_capacity"], marginal_cost=data_p["SMR_marginal_cost"], carrier="H2")
n4.add("Load", "Demand_Pancevo", bus="Serbia_Pancevo_H2", p_set=pd.Series(data_p["H2_demand"], index=n4.snapshots), carrier="H2")

# Unified Croatia Zone
n4.add("Bus", "Croatia_Total_H2", carrier="H2")

# Aggregate Demands and Capacities
demand_total = pd.Series(final_zone_data["Croatia_Rijeka"]["H2_demand"], index=n4.snapshots) + \
               pd.Series(final_zone_data["Croatia_Kutina"]["H2_demand"], index=n4.snapshots)

# Adding the two SMRs as separate supply links to the single bus
n4.add("Link", "SMR_Rijeka_Zonal", bus0="Gas_Grid", bus1="Croatia_Total_H2", p_nom=final_zone_data["Croatia_Rijeka"]["SMR_capacity"], marginal_cost=final_zone_data["Croatia_Rijeka"]["SMR_marginal_cost"], carrier="H2")
n4.add("Link", "SMR_Kutina_Zonal", bus0="Gas_Grid", bus1="Croatia_Total_H2", p_nom=final_zone_data["Croatia_Kutina"]["SMR_capacity"], marginal_cost=final_zone_data["Croatia_Kutina"]["SMR_marginal_cost"], carrier="H2")

n4.add("Load", "Croatia_Total_Demand", bus="Croatia_Total_H2", p_set=demand_total, carrier="H2")
n4.add("Generator", "Emergency_H2_Croatia", bus="Croatia_Total_H2", p_nom=1000, marginal_cost=1000, carrier="H2")


# Sanitize to define carriers and remove warnings
n4.sanitize()

# Explicitly set include_objective_constant to False
n4.optimize(solver_name='highs', include_objective_constant=False)
print(f"Case 4 (Zonal) Total System Cost: {n4.objective:.2f} EUR/hour")


Case 4 (Zonal) Total System Cost: 62900.00 EUR/hour


In [ ]:
print("\n--- Case 4: Zonal Croatia Balance & Prices ---")
gen_p = n4.generators_t.p.fillna(0)
link_p1 = n4.links_t.p1.fillna(0)

balance_n4 = pd.DataFrame({
    'Rijeka_SMR_Prod': -link_p1['SMR_Rijeka_Zonal'],
    'Kutina_SMR_Prod': -link_p1['SMR_Kutina_Zonal'],
    'Total_Demand': n4.loads_t.p['Croatia_Total_Demand'],
    'Marginal_Price': n4.buses_t.marginal_price['Croatia_Total_H2']
}, index=n4.snapshots)

display(balance_n4.round(2))


--- Case 4: Zonal Croatia Balance & Prices ---


,Rijeka_SMR_Prod,Kutina_SMR_Prod,Total_Demand,Marginal_Price
snapshot,,,,
0,210.0,-0.0,210.0,35.0
1,210.0,-0.0,210.0,55.0
2,160.0,-0.0,160.0,65.0
3,160.0,-0.0,160.0,45.0


In [ ]:
print("\n--- Marginal Prices at H2 Buses Per Snapshot (EUR/MWh) - Case 4 (Zonal Modeol) ---")
print("------------------------------------------------------------------\n")

# Accessing n4 marginal prices specifically for H2 buses
h2_prices_n4 = n4.buses_t.marginal_price.filter(regex='_H2')
h2_prices_n4.columns = h2_prices_n4.columns.str.replace('_H2', '')

display(h2_prices_n4.round(2))


--- Marginal Prices at H2 Buses Per Snapshot (EUR/MWh) - Case 4 (Zonal Modeol) ---
------------------------------------------------------------------



name,Serbia_Pancevo,Croatia_Total
snapshot,,
0,55.0,35.0
1,55.0,55.0
2,55.0,65.0
3,55.0,45.0



**1. Key Parameters:**
* **Zone Structure:** Unified Croatia Bus (Rijeka + Kutina capacities and demands aggregated).
* **Total SMR Capacity:** 420 MW (300 MW Rijeka + 120 MW Kutina).
* **Peak Zonal Demand:** 210 MW.

**2. Optimization Results:**
* **Total System Cost:** 62,900.00 EUR/hour.
* **Equivalence to Case 1:** The cost is identical to Case 1, proving that a 300 MW pipeline is sufficient to simulate a single-zone environment without any internal bottlenecks.

**3. H2 Balance:**
* **Production Merit Order:** The optimizer utilizes the **Rijeka SMR** (marginal cost: 5 + gas) to satisfy the entire Croatian demand.
* **Kutina Asset:** The Kutina SMR (marginal cost: 15 + gas) is kept in reserve and produces **0.0 MW** because it is out-competed by the cheaper coastal supply.

**In Conclusion**: Case 4 demonstrates the 'Economic Optimum.' By treating the country as a single zone, the system achieves the lowest possible operational cost by maximizing the use of the most efficient production assets. This case serves as a benchmark for infrastructure planning: any infrastructure (like the pipeline in Case 2) that fails to allow this level of integration results in a 'congestion rent' or higher system-wide costs.

**Cost Equivalence**: The total system cost matches Case 1 exactly (62,900.00 EUR/hour). This proves that with sufficient infrastructure (300MW pipeline), a nodal system behaves identically to a single zonal market.

**Dispatch Efficiency**: The optimizer uses 100% of the cheaper Rijeka SMR capacity before even touching the Kutina SMR. Because total demand (max 210MW) is less than Rijeka's capacity (300MW), the Kutina SMR remains at 0 production.

**Price Convergence**: There is a single, unified price for hydrogen across all of Croatia, determined by the marginal cost of the cheapest available unit (Rijeka SMR + Gas cost).

## Final Summary & Cumulative Comparison



This table compares the economic performance of the four scenarios based on the defined 4-snapshot demand and price profiles.

| Case | Description | Total System Cost (EUR/hr) | Avg. H2 Price Rijeka (EUR/MWh) | Avg. H2 Price Kutina (EUR/MWh) | Avg. H2 Price Pancevo (EUR/MWh) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Case 1** | Integrated (300MW Pipeline) | 62,900.00 | 50.00 | 50.00 | 55.00 |
| **Case 2** | Constrained (80MW Pipeline) | 64,700.00 | 50.00 | 60.00 | 55.00 |
| **Case 3** | Islanded (0MW Pipeline) | 105,700.00 | 50.00 | 532.50* | 55.00 |
| **Case 4** | Zonal (Unified Croatia) | 62,900.00 | 50.00** | 50.00** | 55.00 |

\* *Kutina average price in Case 3 is skewed high by 1,000 EUR/MWh peak prices during supply deficits.*
\** *In Case 4, Croatia is a single zone with a unified price.*